# Phase 4 -- Merge LoRA+DoRA into the base model and quantize to GGUF

Runs on **Google Colab specifically -- not Kaggle** for this notebook. Merging requires
the full bf16 base model (~16.4GB), which doesn't fit this machine's 8GB RAM / 4GB VRAM,
so this has to run on a cloud GPU either way -- but the GGUF conversion step also needs
the merged 16-bit model (~16GB) **and** the F16 GGUF output (~16.4GB) on disk
*simultaneously* while the conversion tool reads one and writes the other. That's ~32GB
of peak disk usage, which structurally cannot fit Kaggle's fixed 20GB limit no matter how
aggressively intermediates are cleaned up before/after -- the peak happens *during* the
conversion itself. A first attempt at this notebook tried exactly that (delete-after-use
cleanup) and still failed on Kaggle for this reason. Colab's disk (commonly reported as
60-110GB on the free tier, though Google doesn't officially publish or guarantee a
number) comfortably covers the ~32GB peak -- confirmed live by this notebook's own
`df -h` checks at each stage, not just assumed.

This is the last cloud-GPU step in the project: the *output* of this notebook (a single
GGUF file) is what actually runs locally for the demo, per the hardware contract in
`docs/blueprint.md`.

## Locked decisions (options-first proposal, confirmed before this notebook was written)

- **Quantization: `Q4_K_M`** (~5.03GB, ~98% quality retention vs. full precision) --
  chosen over smaller quants (Q3_K_S, ~3.77GB) that would fit fully in 4GB VRAM,
  because Q3-tier quality loss risks the structured-JSON reliability that took three
  retrain rounds to earn (see `docs/eval-report.md`). Confirmed: no Q4-tier GGUF fits
  fully in 4GB VRAM alone (5.03GB > 4.0GB) -- **partial GPU offload is required at
  inference time**, handled locally by Ollama's automatic VRAM detection, not here.
- **Adapter merged: the shipped model**, `qwen3-8b-automotive-complaint-lora-FINAL`
  (== v2 from Phase 3 -- solved `severity: high` detection while keeping the best
  `safety_risk` recall; v3 is NOT used here, see `docs/eval-report.md` for why).

## What this notebook does NOT do

It does not run the GGUF file (that happens locally via Ollama) and does not build the
Streamlit demo (that's the final step, done locally once this file is downloaded).

## Upload checklist

Upload these files from `models/qwen3-8b-automotive-complaint-lora-FINAL/` when the
upload prompt appears in the next cell: `adapter_config.json`, `adapter_model.safetensors`,
`tokenizer.json`, `tokenizer_config.json`, `chat_template.jinja`, `README.md`. Skip
`optimizer.pt`/`scheduler.pt`/`scaler.pt`/`rng_state.pth`/`training_args.bin` --
training-resumption state, not needed here.

**Disk note:** this notebook needs real headroom, not just "some" -- the merge (~16GB)
and the F16 GGUF conversion output (~16.4GB) coexist on disk *simultaneously* during
conversion (~32GB peak), before dropping back down once the merged folder is deleted.
This is why this notebook runs on Colab specifically, not Kaggle -- see the note at the
top for the full reasoning.

In [ ]:
%%capture
!pip install unsloth

## 1. Load the shipped adapter (base model + merged FINAL adapter)

In [ ]:
import os

if not (os.path.exists("adapter_config.json") and os.path.exists("adapter_model.safetensors")):
    try:
        from google.colab import files
        print("Upload the FINAL adapter's inference files now (adapter_config.json, "
              "adapter_model.safetensors, tokenizer.json, tokenizer_config.json, "
              "chat_template.jinja, README.md):")
        files.upload()
    except ImportError:
        raise RuntimeError(
            "adapter_config.json / adapter_model.safetensors not found, and "
            "google.colab isn't available. This notebook is Colab-specific -- Kaggle's "
            "fixed 20GB disk can't fit this pipeline's ~32GB peak usage during GGUF "
            "conversion (see the note at the top). Run this on Colab instead."
        )

ADAPTER_DIR = "."
print(f"using adapter from: {ADAPTER_DIR}")

MAX_SEQ_LENGTH = 768  # same as training/eval -- irrelevant to GGUF export itself, but
                       # needed to load the model at all

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    dtype = None,
)

## 2. Merge, convert, and quantize to GGUF (Q4_K_M) -- staged manually for disk space

The first version of this notebook called Unsloth's single `save_pretrained_gguf()`
convenience method, which ran out of disk on Kaggle's 20GB limit. Root cause: three
large copies of the model can coexist simultaneously under that single call -- the
merged 16-bit model (~16GB), a re-downloaded copy of the base model in Hugging Face's
cache (~16GB), and the F16 GGUF intermediate being written (~16.4GB) -- which blows past
20GB before quantization even starts. This is a known, previously-reported issue with
this exact call on Kaggle-sized disks, not something specific to this model.

Checked Unsloth's own source for a way to skip or stream past the F16 intermediate: it's
a required step of the underlying conversion tool (`convert_hf_to_gguf.py`), not
optional, and cleanup of it only happens *after* quantization succeeds -- which never
happens here since disk runs out during the F16 write itself, before that cleanup logic
ever runs.

Fixed by breaking the single call into explicit stages below, deleting each large
intermediate as soon as it's no longer needed, with disk space printed before every
stage that could plausibly run out of room.

In [ ]:
### Stage 2a: merge the adapter into the base weights, saved as 16-bit

MERGED_DIR = "qwen3-8b-automotive-complaint-merged-16bit"

model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method = "merged_16bit")
print(f"merged model saved to {MERGED_DIR}")

print()
print("disk space after merge:")
!df -h .

In [ ]:
### Stage 2b: free the Hugging Face download cache -- no longer needed now the merged
### model above is saved separately on disk. This is the "re-downloaded base model
### copy" that contributed to the original out-of-disk failure.

import shutil, os

hf_cache = os.path.expanduser("~/.cache/huggingface/hub")
if os.path.exists(hf_cache):
    shutil.rmtree(hf_cache, ignore_errors=True)
    print(f"deleted {hf_cache}")
else:
    print(f"{hf_cache} not present, nothing to clean up")

print()
print("disk space after cache cleanup:")
!df -h .

In [ ]:
### Stage 2c: build llama.cpp's quantize tool.
###
### Checked Unsloth's own install_llama_cpp() source directly rather than assuming: it
### does NOT download a prebuilt binary -- it also compiles from source, same as here.
### The difference is it uses an explicit, capped -j{n_jobs} job count, not an
### unrestricted default. An unrestricted `-j` here spawned as many parallel cc1plus
### processes as Colab's core count, and llama.cpp's heavy template-based C++ files
### (llama-model.cpp, the per-architecture model files, etc.) are memory-hungry enough
### per compile unit that this OOM-killed the build (many "Killed signal terminated
### program cc1plus" errors). Fixed by capping to -j 2, matching Unsloth's own approach
### of an explicit, conservative job count rather than "as many as possible."
###
### Also frees a bit of memory first -- the loaded model isn't needed again in this
### session (already merged to disk, in Stage 2a). Guarded with globals() check so this
### cell is safe to rerun even if model/tokenizer were already freed.

import os, gc, torch

for _name in ("model", "tokenizer"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!cmake llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build --config Release -j 2 --target llama-quantize

QUANTIZE_BIN = "llama.cpp/build/bin/llama-quantize"
if not os.path.exists(QUANTIZE_BIN):
    raise RuntimeError(
        f"{QUANTIZE_BIN} not found after build -- check the build output above for errors. "
        f"If it's another OOM ('Killed signal terminated program cc1plus'), try -j 1 "
        f"instead of -j 2 in the cmake --build command above."
    )
print(f"\nllama-quantize built OK: {QUANTIZE_BIN}")

In [ ]:
### Stage 2d: convert the merged 16-bit model to F16 GGUF.
###
### Bug fixed here: the previous version only checked os.path.exists(F16_PATH), which
### is true even for a truncated/corrupt file -- a prior run hit an OSError mid-write
### (disk full) that left a 4.26GB file where a ~16.4GB one was expected, and the old
### check treated that as success and deleted the source model before the failure was
### caught. Fixed by checking the subprocess exit code explicitly AND the output file
### size against the expected range, BEFORE deleting anything.
###
### Self-contained imports (os/shutil/subprocess) so this cell doesn't depend on
### earlier cells having run first in the same session -- a real bug found in an
### earlier version of this notebook (a later cell used os.path.exists() with no local
### import at all).

import os, shutil, subprocess

F16_PATH = "qwen3-8b-automotive-complaint-f16.gguf"
EXPECTED_F16_GB = 16.39  # Qwen3-8B at F16 -- same byte width as the BF16 reference size

result = subprocess.run(
    ["python", "llama.cpp/convert_hf_to_gguf.py", MERGED_DIR, "--outtype", "f16", "--outfile", F16_PATH],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])

if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError(
        f"F16 conversion failed (exit code {result.returncode}). "
        f"NOT deleting {MERGED_DIR} -- see the error output above, check disk space, and retry."
    )

if not os.path.exists(F16_PATH):
    raise RuntimeError(
        f"Conversion reported exit code 0 but {F16_PATH} doesn't exist. "
        f"NOT deleting {MERGED_DIR}."
    )

f16_size_gb = os.path.getsize(F16_PATH) / 1e9
if f16_size_gb < EXPECTED_F16_GB * 0.9:
    os.remove(F16_PATH)  # delete the corrupt/truncated file so nothing downstream can accidentally use it
    raise RuntimeError(
        f"F16 file is only {f16_size_gb:.2f}GB, expected ~{EXPECTED_F16_GB}GB (below 90% "
        f"threshold -- looks truncated, likely a disk-space failure mid-write even though "
        f"the process exited 0). Deleted the corrupt file. NOT deleting {MERGED_DIR} since "
        f"conversion didn't actually succeed -- check disk space above and retry."
    )

print(f"F16 GGUF written and verified: {F16_PATH} ({f16_size_gb:.2f}GB, matches expected range)")

# Only delete the merged model now that the F16 file is confirmed complete and correctly sized.
shutil.rmtree(MERGED_DIR, ignore_errors=True)
print(f"deleted {MERGED_DIR} (confirmed no longer needed -- {F16_PATH} verified complete)")

print()
print("disk space before quantization:")
!df -h .

In [ ]:
### Stage 2e: quantize F16 -> Q4_K_M.
###
### Same fix as Stage 2d applied here too, for consistency -- exit code + size check
### before deleting the F16 source, even though this step wasn't the one that failed.
### Self-contained imports, same reasoning as Stage 2d.

import os, subprocess

Q4_PATH = "qwen3-8b-automotive-complaint-Q4_K_M.gguf"  # matches models/Modelfile's expected filename
EXPECTED_Q4_GB = 5.03  # Qwen3-8B Q4_K_M, per Unsloth's own published GGUF repo

result = subprocess.run(
    [QUANTIZE_BIN, F16_PATH, Q4_PATH, "Q4_K_M"],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])

if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError(
        f"Quantization failed (exit code {result.returncode}). "
        f"NOT deleting {F16_PATH} -- see the error output above, check disk space, and retry."
    )

if not os.path.exists(Q4_PATH):
    raise RuntimeError(
        f"Quantization reported exit code 0 but {Q4_PATH} doesn't exist. NOT deleting {F16_PATH}."
    )

q4_size_gb = os.path.getsize(Q4_PATH) / 1e9
if q4_size_gb < EXPECTED_Q4_GB * 0.9:
    os.remove(Q4_PATH)
    raise RuntimeError(
        f"Q4_K_M file is only {q4_size_gb:.2f}GB, expected ~{EXPECTED_Q4_GB}GB (below 90% "
        f"threshold -- looks truncated). Deleted the corrupt file. NOT deleting {F16_PATH} "
        f"since quantization didn't actually succeed -- check disk space above and retry."
    )

print(f"Q4_K_M GGUF written and verified: {Q4_PATH} ({q4_size_gb:.2f}GB, matches expected range)")

os.remove(F16_PATH)
print(f"deleted {F16_PATH} (confirmed no longer needed -- {Q4_PATH} verified complete)")

print()
print("disk space after quantization:")
!df -h .

## 3. Locate, verify, and download the GGUF file

In [ ]:
import os

size_gb = os.path.getsize(Q4_PATH) / 1e9
print(f"GGUF file: {Q4_PATH}")
print(f"size: {size_gb:.2f} GB")

expected_gb = 5.03  # Qwen3-8B Q4_K_M, per Unsloth's own published GGUF repo
if abs(size_gb - expected_gb) > 0.5:
    print(f"FLAG: size differs from the expected ~{expected_gb}GB by more than 0.5GB -- "
          f"double-check the quantization actually applied (q4_k_m, not an accidental "
          f"f16/bf16 export) before downloading.")
else:
    print("size matches the expected Q4_K_M range -- looks correct.")

In [ ]:
from google.colab import files
files.download(Q4_PATH)

## Next

Download the `.gguf` file and bring it back to the local machine. Next step there:
load it with Ollama (partial GPU offload, handled automatically) and confirm it produces
sensible structured JSON on a real complaint before building the Streamlit demo.